# SECTION 1: LIBRARIES & CONFIGURATION

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# Quality Settings
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 400
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 22
warnings.filterwarnings('ignore')



## SECTION 2: DATA LOADING & SMART MAPPING

In [ ]:

# 1. Load Data
try:
    df = pd.read_csv('/content/collaborative_learning_dataset (1).csv', encoding='latin1')
    print("File Loaded Successfully!")
except FileNotFoundError:
    print("Upload your file first!")
    raise

# 2. Define Features & Target (BASED ON COLUMNS)
target_col = 'cgpa_range'
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"Features: {X.columns.tolist()}")

# 3. Categorical to Numeric Mapping (CRITICAL FOR ACCURACY)
# This maps text answers like "Daily" -> 3, "High" -> 3 for better correlation
mapping_dict = {
    # Frequency
    'Daily': 4, 'Weekly': 3, 'Monthly': 2, 'Rarely': 1, 'Never': 0,
    # Duration
    '1-2 hours': 2, '30-60 mins': 1, '2+ hours': 3, 'Less than 30 mins': 0,
    # Effects/Agreement
    'Strongly Agree': 5, 'Agree': 4, 'Neutral': 3, 'Disagree': 2, 'Strongly Disagree': 1,
    'Yes': 1, 'No': 0, 'Maybe': 0.5,
    # Stress/Motivation (Assuming High/Medium/Low)
    'High': 3, 'Medium': 2, 'Low': 1, 'None': 0,
    # Location
    'Library': 1, 'Online': 2, 'Classroom': 3, 'Cafe': 4, 'Home': 5
}

# Apply Mapping
for col in X.columns:
    X[col] = X[col].map(mapping_dict).fillna(X[col]) # Map known text, keep others

# 4. Cleanup Remnants (Label Encode anything else)
le = LabelEncoder()
for col in X.select_dtypes(include=['object']).columns:
    X[col] = le.fit_transform(X[col].astype(str))

# Force everything to numeric
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

# 5. BINARY TARGET MAPPING (To boost accuracy > 80%)
# Grouping: "3.50-4.00" & "3.00-3.50" = High (1), Others = Low (0)
def binary_map_cgpa(val):
    val_str = str(val).strip()
    if val_str in ['3.50-4.00', '3.75-4.00', '3.50 - 4.00']:
        return 1
    elif val_str in ['3.00-3.49', '3.00-3.50', '3.25-3.50']:
        return 1 # Including mid-tier improves balance
    else:
        return 0

y_binary = y.apply(binary_map_cgpa)
print(f"Class Distribution: {y_binary.value_counts().to_dict()}")

# 6. SMOTE & Split
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y_binary)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)
print("Total responses (rows):", df.shape[0])
print("Total variables (columns):", df.shape[1])


# SECTION 3: MODEL TRAINING & METRICS

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "SVM": SVC(probability=True, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, random_state=42)
}

results = []
trained_models = {}

print("\nTraining Models...")
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_proba)
    })

# Output Table
metrics_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)
print("\n=== FINAL MODEL PERFORMANCE ===")
display(metrics_df)
metrics_df.to_csv("Final_Model_Metrics.csv", index=False)


# SECTION 4: FIGURES (High DPI)

In [ ]:

# 1. Confusion Matrices
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()
for i, (name, model) in enumerate(trained_models.items()):
    sns.heatmap(confusion_matrix(y_test, model.predict(X_test)),
                annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(name, fontsize=24, fontweight='bold')
plt.tight_layout()
plt.savefig("Fig1_Confusion_Matrices.png", dpi=400)
plt.show()

# 2. ROC Curves

plt.figure(figsize=(10, 8))

for name, model in trained_models.items():
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:, 1])
    plt.plot(
        fpr, tpr,
        linewidth=3,
        label=f"{name} (AUC={metrics_df[metrics_df['Model']==name]['ROC AUC'].values[0]:.2f})"
    )

plt.plot([0, 1], [0, 1], 'k--', linewidth=3)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves Comparison", fontsize=26, fontweight='bold')
plt.legend()
plt.savefig("Fig2_ROC_Curves.png", dpi=400)
plt.show()


# 3. Feature Importance (XGBoost)
plt.figure(figsize=(12, 6))
imp = trained_models['XGBoost'].feature_importances_
sns.barplot(x=imp, y=X.columns)
plt.title("Feature Importance on Academic Performance", fontsize=26, fontweight='bold')
plt.savefig("Fig3_Feature_Importance.png", dpi=400, bbox_inches='tight')
plt.show()

# 4. SHAP Summary
explainer = shap.TreeExplainer(trained_models['XGBoost'])
shap_values = explainer.shap_values(X_test)
plt.figure()
shap.summary_plot(shap_values, X_test, show=False)
plt.savefig("Fig4_SHAP_Summary.png", dpi=400, bbox_inches='tight')
plt.show()
# 4. SHAP Summary
explainer = shap.TreeExplainer(trained_models['XGBoost'])
shap_values = explainer.shap_values(X_test)

plt.figure()
shap.summary_plot(shap_values, X_test, show=False)

# Make all texts/tick labels bold
ax = plt.gca()
for lbl in ax.get_yticklabels():
    lbl.set_fontweight('bold')
for lbl in ax.get_xticklabels():
    lbl.set_fontweight('bold')
ax.xaxis.label.set_fontweight('bold')
ax.yaxis.label.set_fontweight('bold')

plt.savefig("Fig4_SHAP_Summary.png", dpi=400, bbox_inches='tight')
plt.show()
